# `nn.py` — Full Matrix VJP (`@`), Shape Operators & Neural Network Modules

**Purpose:** This notebook contains the complete code of `src/tensorgrad/nn.py`, divided into:
1. Imports and Core Dependencies
2. Matrix Multiplication VJP (`matmul`, `rmatmul`) and Shape Operators (`reshape`, `transpose`)
3. Base `Module` Container and `Linear` Layer with He Initialization
4. Activation Layers (`ReLU`, `Tanh`), `Sequential`, and `MLP`
5. Full Main Execution & Gradient Verification

## 1. Imports and Setup

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve() / "src"))

from typing import Iterable, List, Optional, Sequence, Tuple, Union
import numpy as np

from tensorgrad.tensor import ArrayLike, Tensor, _unbroadcast, numerical_grad_check
print("Imported core Tensor primitives.")

Imported core Tensor primitives.


## 2. Matrix Multiplication (`@`) VJP and Shape Transformations (`reshape`, `transpose`)
Implements $Z = X \mathbin{@} Y$ with $\frac{\partial L}{\partial X} = G Y^T$ and $\frac{\partial L}{\partial Y} = X^T G$, alongside `reshape` and `transpose` (using inverse permutation `np.argsort(axes)`), and binds them onto `Tensor`.

In [2]:
def matmul(x: Tensor, y: Union[Tensor, ArrayLike]) -> Tensor:
    """
    Matrix multiplication: Z = X @ Y.
    Backward VJP:
        dL/dX = G @ Y^T
        dL/dY = X^T @ G
    """
    y_tensor = x._ensure_tensor(y)
    out = Tensor(
        x.data @ y_tensor.data,
        requires_grad=x.requires_grad or y_tensor.requires_grad,
        _children=(x, y_tensor),
        _op="@",
    )

    def _backward() -> None:
        if x.requires_grad:
            y_t = np.swapaxes(y_tensor.data, -1, -2)
            x.grad = x.grad + _unbroadcast(out.grad @ y_t, x.shape)
        if y_tensor.requires_grad:
            x_t = np.swapaxes(x.data, -1, -2)
            y_tensor.grad = y_tensor.grad + _unbroadcast(x_t @ out.grad, y_tensor.shape)

    out._backward = _backward
    return out


def rmatmul(x: Tensor, other: ArrayLike) -> Tensor:
    return Tensor._ensure_tensor(other) @ x


def reshape(x: Tensor, *new_shape: Union[int, Tuple[int, ...]]) -> Tensor:
    """Reshape tensor: Z = X.reshape(new_shape) -> Backward: G.reshape(X.shape)."""
    if len(new_shape) == 1 and isinstance(new_shape[0], (tuple, list)):
        target_shape = tuple(new_shape[0])
    else:
        target_shape = tuple(int(s) for s in new_shape)

    out = Tensor(
        x.data.reshape(target_shape),
        requires_grad=x.requires_grad,
        _children=(x,),
        _op="reshape",
    )

    def _backward() -> None:
        if x.requires_grad:
            x.grad = x.grad + out.grad.reshape(x.shape)

    out._backward = _backward
    return out


def transpose(x: Tensor, axes: Optional[Sequence[int]] = None) -> Tensor:
    """
    Permute tensor axes: Z = X.transpose(axes).
    Backward applies the inverse permutation `np.argsort(axes)`.
    """
    axes_tuple = None if axes is None else tuple(axes)
    out = Tensor(
        x.data.transpose(axes_tuple),
        requires_grad=x.requires_grad,
        _children=(x,),
        _op="transpose",
    )

    def _backward() -> None:
        if x.requires_grad:
            if axes_tuple is None:
                inv_grad = out.grad.transpose()
            else:
                inv_axes = tuple(int(i) for i in np.argsort(axes_tuple))
                inv_grad = out.grad.transpose(inv_axes)
            x.grad = x.grad + inv_grad

    out._backward = _backward
    return out


Tensor.__matmul__ = matmul
Tensor.__rmatmul__ = rmatmul
Tensor.matmul = matmul
Tensor.reshape = reshape
Tensor.transpose = transpose
Tensor.T = property(lambda self: transpose(self))

# Quick check of @ and .T
X_m = Tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
Y_m = Tensor([[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]])
Z_m = X_m @ Y_m
Z_m.sum().backward()
print("Z = X @ Y:\n", Z_m.data)
print("X.grad:\n", X_m.grad)

Z = X @ Y:
 [[ 4. -1.]
 [10. -1.]]
X.grad:
 [[1. 1. 0.]
 [1. 1. 0.]]


## 3. `Module` Base Class and `Linear` Layer (He Initialization)
`Module` recursively collects trainable `Tensor` parameters and resets gradients via `zero_grad()`. `Linear` initializes weights with $\sigma = \sqrt{2 / d_{\text{in}}}$.

In [3]:
class Module:
    """Base class for all neural network layers and models."""

    def parameters(self) -> List[Tensor]:
        params: List[Tensor] = []
        for val in self.__dict__.values():
            if isinstance(val, Tensor) and val.requires_grad:
                params.append(val)
            elif isinstance(val, Module):
                params.extend(val.parameters())
            elif isinstance(val, (list, tuple)):
                for item in val:
                    if isinstance(item, Tensor) and item.requires_grad:
                        params.append(item)
                    elif isinstance(item, Module):
                        params.extend(item.parameters())
        return params

    def zero_grad(self) -> None:
        """Reset gradients of all trainable parameters to zero."""
        for p in self.parameters():
            p.zero_grad()

    def forward(self, *args, **kwargs) -> Tensor:
        raise NotImplementedError

    def __call__(self, *args, **kwargs) -> Tensor:
        return self.forward(*args, **kwargs)


class Linear(Module):
    """
    Fully connected linear layer: Y = X @ W + b.
    Initialized with He (Kaiming) normal scaling: std = sqrt(2 / in_features).
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        rng: Optional[np.random.Generator] = None,
    ) -> None:
        self.in_features = int(in_features)
        self.out_features = int(out_features)
        gen = rng if rng is not None else np.random.default_rng()

        std = np.sqrt(2.0 / float(self.in_features))
        w_data = gen.normal(loc=0.0, scale=std, size=(self.in_features, self.out_features))
        self.weight = Tensor(w_data, requires_grad=True)
        self.bias: Optional[Tensor] = (
            Tensor(np.zeros((self.out_features,), dtype=np.float64), requires_grad=True)
            if bias
            else None
        )

    def forward(self, x: Tensor) -> Tensor:
        out = x @ self.weight
        if self.bias is not None:
            out = out + self.bias
        return out

    def __repr__(self) -> str:
        return (
            f"Linear(in_features={self.in_features}, "
            f"out_features={self.out_features}, bias={self.bias is not None})"
        )


lin_demo = Linear(4, 6, rng=np.random.default_rng(0))
print("Linear layer:", lin_demo)
print("Weight first 2 rows:\n", np.round(lin_demo.weight.data[:2], 4))

Linear layer: Linear(in_features=4, out_features=6, bias=True)
Weight first 2 rows:
 [[ 0.0889 -0.0934  0.4528  0.0742 -0.3788  0.2557]
 [ 0.9221  0.6697 -0.4976 -0.8948 -0.4407  0.0292]]


## 4. Activation Modules (`ReLU`, `Tanh`), `Sequential`, and `MLP`
Constructs arbitrary multi-layer perceptrons with non-linear hidden activations and a linear output head.

In [4]:
class ReLU(Module):
    """Rectified Linear Unit activation layer: max(0, X)."""

    def forward(self, x: Tensor) -> Tensor:
        return x.relu()

    def __repr__(self) -> str:
        return "ReLU()"


class Tanh(Module):
    """Hyperbolic tangent activation layer: tanh(X)."""

    def forward(self, x: Tensor) -> Tensor:
        return x.tanh()

    def __repr__(self) -> str:
        return "Tanh()"


class Sequential(Module):
    """Sequential container chaining an ordered list of modules."""

    def __init__(self, *layers: Module) -> None:
        self.layers: List[Module] = list(layers)

    def forward(self, x: Tensor) -> Tensor:
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

    def __repr__(self) -> str:
        inner = ", ".join(repr(layer) for layer in self.layers)
        return f"Sequential({inner})"


class MLP(Module):
    """
    Multi-Layer Perceptron:
        in_dim -> hidden_dims[0] -> ... -> out_dim
    Applies non-linear activations between hidden layers and leaves the final layer linear.
    """

    def __init__(
        self,
        in_dim: int,
        hidden_dims: Iterable[int],
        out_dim: int,
        activation: str = "relu",
        rng: Optional[np.random.Generator] = None,
    ) -> None:
        gen = rng if rng is not None else np.random.default_rng()
        dims = [int(in_dim)] + [int(h) for h in hidden_dims] + [int(out_dim)]

        layers: List[Module] = []
        for i in range(len(dims) - 1):
            layers.append(Linear(dims[i], dims[i + 1], bias=True, rng=gen))
            if i < len(dims) - 2:
                if activation.lower() == "relu":
                    layers.append(ReLU())
                elif activation.lower() == "tanh":
                    layers.append(Tanh())
                else:
                    raise ValueError(f"Unsupported activation: {activation}")

        self.net = Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

    def __repr__(self) -> str:
        return f"MLP({self.net})"

## 5. Main Execution Block
Runs the full `nn.py` verification suite and prints output shapes and parameter counts.

In [5]:
print("=" * 72)
print("tensorgrad — Module 2: Matrix VJP (@), Linear Layer & MLP")
print("=" * 72)

rng_demo = np.random.default_rng(42)

A = Tensor(rng_demo.normal(size=(4, 4)))
B = Tensor(rng_demo.normal(size=(4, 4)))
C = (A @ B).reshape(2, 8).T
print(f"[1] A.shape={A.shape}, B.shape={B.shape} -> (A@B).reshape(2,8).T.shape={C.shape}")

mlp = MLP(in_dim=4, hidden_dims=[16, 8], out_dim=2, rng=rng_demo)
X_batch = Tensor(rng_demo.normal(size=(8, 4)))
logits = mlp(X_batch)
total_params = sum(p.size for p in mlp.parameters())
print(f"\n[2] MLP Architecture     : {mlp}")
print(f"    Input X.shape        : {X_batch.shape} -> Output logits.shape: {logits.shape}")
print(f"    First 3 logits       :\n{np.round(logits.data[:3], 4)}")
print(f"    Total Parameters     : {total_params} (6 Tensors: 3 weights, 3 biases)")
print("=" * 72)

tensorgrad — Module 2: Matrix VJP (@), Linear Layer & MLP
[1] A.shape=(4, 4), B.shape=(4, 4) -> (A@B).reshape(2,8).T.shape=(8, 2)

[2] MLP Architecture     : MLP(Sequential(Linear(in_features=4, out_features=16, bias=True), ReLU(), Linear(in_features=16, out_features=8, bias=True), ReLU(), Linear(in_features=8, out_features=2, bias=True)))
    Input X.shape        : (8, 4) -> Output logits.shape: (8, 2)
    First 3 logits       :
[[ 0.7102  1.2339]
 [-2.0911  0.3898]
 [-1.978   0.4195]]
    Total Parameters     : 234 (6 Tensors: 3 weights, 3 biases)


## Conclusion
The full `nn.py` module provides exact matrix VJPs, shape operators, He-initialized linear layers, and composable `MLP` architectures.